# 05 Fine-tuned pretrained ResNet

Functions: `src/piece_of_crop/resnet.py`, and the training loop from `src/piece_of_crop/train.py` (to be written after the design is agreed).

**Purpose.** Fine-tune a ResNet pretrained on ImageNet on RGB and on all 13 bands, with the same split, settings and seeds as the CNN.

**Reads.** The split file and band statistics from notebook 02.
**Saves.** Test predictions as `results/predictions/resnet_<input>_seed<n>.csv` and the training curves in `results/figures/`. Model weights stay local and are not committed.

**Why.** Fine-tuning takes the feature extraction layers of a pretrained CNN, replaces the classifier and retrains some or all layers (lecture_12_finetuning and transfer learning.pdf, slides 5 and 8). The course notebook compares training from scratch, feature extraction and full fine-tuning with a smaller learning rate for the backbone (lecture_12_finetuning_live_WITH_RESULTS 1.pdf, section 4). The ImageNet weights were learned on RGB photos, so for 13 bands the first layer must be rebuilt, which gives the RGB version a head start. This is stated as a limitation.

## Setup: shared data for every model

This cell is the same in notebooks 03, 04 and 05, so every model starts from exactly the same data. It reads the split (`results/split.csv`) and the normalisation statistics (`results/band_stats.csv`) saved by notebook 02, and opens the cache of all 27,000 patches. The first notebook you run on your computer builds the cache once (`data/eurosat_cache.npy`, about 2.9 GB, not in Git); after that this cell only loads it. Do not change this cell: if you need something different, ask Estelle first, because a different split or different statistics would make the models impossible to compare.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from piece_of_crop import get_cache, make_loaders, read_split

SEED = 0
project_root = Path.cwd().parents[0]
data_dir = project_root / "data" / "EuroSAT_MS"
results_dir = project_root / "results"
figures_dir = results_dir / "figures"

split = read_split(results_dir / "split.csv")
band_stats = pd.read_csv(results_dir / "band_stats.csv")
cache = get_cache(split, data_dir, project_root / "data" / "eurosat_cache.npy")
print("Patches per split:", split["split"].value_counts().to_dict())